In [0]:
import os
import sys
import shutil
import subprocess
import errno
from contextlib import contextmanager
from datetime import datetime
from pyspark.sql import functions as F

CAT = 'machine_health.prod'
AUDIT = CAT + '.pipeline_run_audit'


# ============================================================
# FINAL AUDIT COLUMN LIST
# Must match the ClairAI metadata requirements
# ============================================================

COLS = [
    'workspace_id',
    'job_name',
    'asset_type',
    'job_id',
    'job_run_id',
    'task_run_id',
    'task_key',
    'machine_type',
    'attempt_number',
    'status',
    'start_time',
    'end_time',
    'cluster_id',
    'error_message',
    'run_page_url',
    'task_run_page_url',
    'table_name',
    'operation',
    'conflicting_job_run_id',
    'conflicting_run_page_url',
    'read_version',
    'conflicting_version'
]


# ============================================================
# WIDGET DECLARATION
# ============================================================

def declare_widgets(**extra):

    base = dict(
        machine_type='',
        job_id='',
        job_run_id='',
        task_run_id='',
        task_key='',
        execution_count='1',
        workspace_id='',
        job_name='',
        inject_fault='none',
        fault_task=''
    )

    base.update(extra)

    for key, value in base.items():
        dbutils.widgets.text(
            key,
            str(value)
        )


# ============================================================
# MACHINE SCOPE
# ============================================================

def scope(machine_type):
    return f"machine_type = '{machine_type}'"


# ============================================================
# RUNTIME METADATA
#
# workspace_id:
#   Retrieved directly from Databricks runtime context.
#
# job_name:
#   Retrieved from the Databricks Jobs API using job_id.
# ============================================================

def runtime_metadata():

    context = (
        dbutils
        .notebook
        .entry_point
        .getDbutils()
        .notebook()
        .getContext()
    )

    workspace_id = context.workspaceId().get()

    job_id = dbutils.widgets.get('job_id')

    from databricks.sdk import WorkspaceClient

    workspace = WorkspaceClient()

    if job_id:
        job_name = (
            workspace
            .jobs
            .get(int(job_id))
            .settings
            .name
        )
    else:
        job_name = ''

    return workspace_id, job_name


# ============================================================
# AUDIT CONTEXT MANAGER
#
# Creates one audit record for each task attempt.
#
# SUCCESS:
#   Record is written after the task completes.
#
# FAILED:
#   Error is captured, record is written, then exception
#   is re-raised so Databricks still sees the task as failed.
#
# IMPORTANT:
# read_version and conflicting_version start as NULL.
# Case 2 apply_health_rules populates them when a
# ConcurrentAppendException occurs.
# ============================================================

@contextmanager
def audited():

    get = dbutils.widgets.get

    workspace_id, job_name = runtime_metadata()

    rec = dict.fromkeys(COLS)

    rec.update(
        workspace_id=workspace_id,
        job_name=job_name,
        asset_type=get('machine_type'),
        job_id=get('job_id'),
        job_run_id=get('job_run_id'),
        task_run_id=get('task_run_id'),
        task_key=get('task_key'),
        machine_type=get('machine_type'),

        attempt_number=int(
            get('execution_count') or '1'
        ),

        status='SUCCESS',
        start_time=datetime.utcnow()
    )

    try:

        yield rec

    except Exception as e:

        rec.update(
            status='FAILED',
            error_message=str(e)[:3000]
        )

        raise

    finally:

        rec['end_time'] = datetime.utcnow()

        # ----------------------------------------------------
        # IMPORTANT
        #
        # Use the physical table schema rather than COLS order.
        #
        # This protects us because ALTER TABLE ADD COLUMNS
        # can append columns physically at the end.
        # ----------------------------------------------------

        table_schema = spark.table(AUDIT).schema

        row = tuple(
            rec[field.name]
            for field in table_schema
        )

        (
            spark
            .createDataFrame(
                [row],
                table_schema
            )
            .write
            .mode('append')
            .saveAsTable(AUDIT)
        )


# ============================================================
# CASE 1
# LOCAL DISK EXHAUSTION / LIBRARY INSTALL FAILURE
# ============================================================

def maybe_disk_fault():

    get = dbutils.widgets.get

    if get('inject_fault') != 'disk':
        return

    if get('fault_task') != get('task_key'):
        return

    fill = '/local_disk0/fill.bin'

    try:

        if not os.path.exists('/local_disk0'):
            return

        free = shutil.disk_usage(
            '/local_disk0'
        ).free

        fill_size = max(
            1,
            free - (50 * 1024 * 1024)
        )

        subprocess.run(
            [
                'fallocate',
                '-l',
                str(fill_size),
                fill
            ],
            check=True
        )

        result = subprocess.run(
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--force-reinstall',
                '--no-cache-dir',
                'scipy'
            ],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:

            raise RuntimeError(
                (
                    result.stdout +
                    result.stderr
                )[-3000:]
            )

    except Exception:

        raise OSError(
            errno.ENOSPC,
            'No space left on device'
        )

    finally:

        if os.path.exists(fill):

            try:
                os.remove(fill)

            except Exception:
                pass